In [0]:
# ============================================================
# BRONZE LAYER: Raw Takeout JSON → Delta Table
# ============================================================
# Goal: Land raw data as-is in a queryable Delta table.
# No cleaning, no transformation. Preserve everything.
# ============================================================

from pyspark.sql.functions import current_timestamp, input_file_name, lit

# Config
RAW_PATH = "/Volumes/youtube_wrapped/bronze/raw_takeout/watch-history.json"
BRONZE_TABLE = "youtube_wrapped.bronze.watch_history_raw"

print(f"Source: {RAW_PATH}")
print(f"Target: {BRONZE_TABLE}")

In [0]:
# Read the JSON file as a Spark DataFrame
# Takeout's watch-history.json is a single JSON array, so we use multiLine=True
df_raw = (
    spark.read
    .option("multiLine", "true")
    .json(RAW_PATH)
)

print(f"Rows read: {df_raw.count():,}")
df_raw.printSchema()

In [0]:
# Look at 5 sample rows to sanity check the data
df_raw.show(5, truncate=80)

In [0]:
# Add ingestion metadata — critical for debugging and lineage
# Note: In Unity Catalog, use _metadata.file_path instead of input_file_name()
df_bronze = (
    df_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", df_raw["_metadata.file_path"])
    .withColumn("_pipeline_run_id", lit("manual-001"))
)

df_bronze.select("_ingested_at", "_source_file", "_pipeline_run_id").show(3, truncate=False)

In [0]:
# Richer version with more file metadata
df_bronze = (
    df_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", df_raw["_metadata.file_path"])
    .withColumn("_source_file_modified_at", df_raw["_metadata.file_modification_time"])
    .withColumn("_pipeline_run_id", lit("manual-001"))
)

df_bronze.select(
    "_ingested_at", 
    "_source_file", 
    "_source_file_modified_at",
    "_pipeline_run_id"
).show(3, truncate=False)

In [0]:
# Write as Delta table in Unity Catalog
(
    df_bronze.write
    .mode("overwrite")  # overwrite is fine for bronze reprocessing
    .option("overwriteSchema", "true")  # allow schema changes on reload
    .saveAsTable(BRONZE_TABLE)
)

print(f"✅ Wrote to {BRONZE_TABLE}")

In [0]:
# Read back from the Delta table to confirm it's queryable
df_check = spark.table(BRONZE_TABLE)

print(f"Rows in bronze table: {df_check.count():,}")
print(f"\nColumns: {df_check.columns}")
print(f"\nSample row:")
df_check.show(1, vertical=True, truncate=80)

In [0]:
%sql
-- Verify the table exists in Unity Catalog
DESCRIBE TABLE youtube_wrapped.bronze.watch_history_raw;

-- How many records?
SELECT COUNT(*) AS total_records FROM youtube_wrapped.bronze.watch_history_raw;

-- What's the date range?
SELECT 
  MIN(time) AS earliest_activity,
  MAX(time) AS latest_activity
FROM youtube_wrapped.bronze.watch_history_raw;